# Ragas로 RAG 평가 이해하기

RAG 품질을 두 평가 영역으로 구분

- **검색(Retrieval)**: 필요한 문서를 찾았는가?
- **생성(Generation)**: 검색 문서에 근거해 질문에 답했는가?

예제 1건과 핵심 지표 4개를 사용한 입문 실습  
Ragas 0.4.3 기준

In [1]:
from dotenv import load_dotenv
load_dotenv()

False

## 1. 평가 데이터

| 필드 | 의미 |
|---|---|
| `user_input` | 사용자 질문 |
| `retrieved_contexts` | 검색된 문서 조각 목록 |
| `response` | RAG가 생성한 답변 |
| `reference` | 사람이 작성한 기준 답변(정답) |

검색 순서: 정답 문서 → 관련은 있지만 불필요한 문서

In [2]:
# 하나의 질문에 대한 RAG 실행 결과와 기준 답변
sample = {
    "user_input": "대한민국의 수도는 어디인가요?",
    "retrieved_contexts": [
        # 첫 번째 문서: 정답에 필요한 정보
        "서울은 대한민국의 수도이다.",
        # 두 번째 문서: 관련 주제지만 정답에는 불필요한 정보
        "부산은 대한민국의 대표적인 항구 도시이다.",
    ],
    "response": "대한민국의 수도는 서울이다.",
    "reference": "대한민국의 수도는 서울이다.",
}

sample

{'user_input': '대한민국의 수도는 어디인가요?',
 'retrieved_contexts': ['서울은 대한민국의 수도이다.', '부산은 대한민국의 대표적인 항구 도시이다.'],
 'response': '대한민국의 수도는 서울이다.',
 'reference': '대한민국의 수도는 서울이다.'}

## 2. 핵심 지표

| 평가 영역 | 지표 | 평가 기준 |
|---|---|---|
| 생성 | `Faithfulness` | 답변이 검색 문서에 근거하는 정도 |
| 생성 | `AnswerRelevancy` | 답변이 질문과 관련된 정도 |
| 검색 | `ContextPrecisionWithReference` | 유용한 문서가 앞쪽에 배치된 정도 |
| 검색 | `ContextRecall` | 필요한 정보를 검색 문서가 포함한 정도 |

일반적인 점수 범위: **0~1**  
높은 점수: 해당 평가 기준을 잘 충족

In [3]:
from openai import AsyncOpenAI
from ragas.embeddings import OpenAIEmbeddings
from ragas.llms import llm_factory

# ascore()가 비동기 평가 메서드이므로 AsyncOpenAI 클라이언트 사용
# API 응답을 기다리는 동안 노트북의 실행 흐름을 차단하지 않음
# 답변과 검색 문서를 판정할 평가용 LLM
evaluator_llm = llm_factory(
    model="gpt-4o-mini",
    client=AsyncOpenAI(),
    temperature=0,
)

# 비동기 평가에서 의미 유사도를 계산할 임베딩 모델
evaluator_embeddings = OpenAIEmbeddings(
    client=AsyncOpenAI(),
    model="text-embedding-3-small",
)

d:\rag_two_branch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

## 3. 지표별 평가

이 셀부터 OpenAI API 사용량 발생

In [ ]:
from ragas.metrics.collections import (
    AnswerRelevancy,
    ContextPrecisionWithReference,
    ContextRecall,
    Faithfulness,
)

# 생성 평가 1: 답변의 내용이 검색 문서에 근거하는지 확인
faithfulness = await Faithfulness(llm=evaluator_llm).ascore(
    user_input=sample["user_input"],
    response=sample["response"],
    retrieved_contexts=sample["retrieved_contexts"],
)

# 생성 평가 2: 답변이 질문에 직접 관련되는지 확인
answer_relevancy = await AnswerRelevancy(
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    strictness=1,
).ascore(
    user_input=sample["user_input"],
    response=sample["response"],
)

# 검색 평가 1: 정답에 유용한 문서가 앞쪽에 있는지 확인
context_precision = await ContextPrecisionWithReference(
    llm=evaluator_llm
).ascore(
    user_input=sample["user_input"],
    reference=sample["reference"],
    retrieved_contexts=sample["retrieved_contexts"],
)

# 검색 평가 2: 정답에 필요한 정보가 검색 문서에 있는지 확인
context_recall = await ContextRecall(llm=evaluator_llm).ascore(
    user_input=sample["user_input"],
    reference=sample["reference"],
    retrieved_contexts=sample["retrieved_contexts"],
)

## 4. 결과 확인

In [ ]:
import pandas as pd

# 네 지표의 점수를 검색과 생성으로 구분해 한 표로 정리
scores = pd.DataFrame(
    [
        {"평가 영역": "생성", "지표": "Faithfulness", "점수": faithfulness.value},
        {"평가 영역": "생성", "지표": "Answer Relevancy", "점수": answer_relevancy.value},
        {"평가 영역": "검색", "지표": "Context Precision", "점수": context_precision.value},
        {"평가 영역": "검색", "지표": "Context Recall", "점수": context_recall.value},
    ]
)

scores

,평가 영역,지표,점수
0,생성,Faithfulness,1.000000
1,생성,Answer Relevancy,0.272911
2,검색,Context Precision,1.000000
3,검색,Context Recall,1.000000


## 5. 점수 변화 실험

한 번에 한 요소만 변경

1. 답변을 `대한민국의 수도는 부산이다.`로 변경 → 생성 점수 확인
2. 서울 문서 삭제 → `Context Recall` 확인
3. 부산 문서를 첫 번째로 이동 → `Context Precision` 확인

LLM 기반 평가 특성상 실행마다 작은 점수 차이 가능